# RAG Evaluation Framework – Prototype 1

This notebook implements a minimal, controlled evaluation framework for
retrieval-augmented generation (RAG) systems.

We begin with a small synthetic corpus to:
- Create a controlled retrieval task
- Establish ground-truth mappings
- Enable systematic evaluation of retrieval accuracy and answer faithfulness

## Environment Setup

This cell loads environment variables and verifies that required configuration

In [18]:
from dotenv import load_dotenv
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


## RAG Pipeline

In this section, we demonstrate a **minimal Retrieval-Augmented Generation (RAG) pipeline** using a controlled corpus and evaluation queries. The pipeline covers the following stages:

1. **Document Embeddings** — Generate vector representations of each document using a configurable embedding model and persist them for later use.  
2. **RAG Retrieval and Querying** — Build a FAISS vector index over the document embeddings and retrieve the most relevant documents for given queries.  
3. **Evaluation** — Compute retrieval accuracy against ground-truth labels to validate the correctness of embeddings, retrieval, and evaluation logic.

The following steps (Step 1 through Step 4) walk through this pipeline in a **reproducible and minimal demonstration**, providing a foundation for later extraction into reusable framework components.


### Step 1 — Define Controlled Corpus and Queries

We define:
- A small document corpus
- A set of queries
- Ground-truth document IDs for retrieval evaluation

In [19]:
import json
from pathlib import Path

# Base data path (relative to notebook)
BASE_DATA_PATH = Path("../data")
RAW_DATA_PATH = BASE_DATA_PATH / "raw"
PROCESSED_DATA_PATH = BASE_DATA_PATH / "processed"

with open(RAW_DATA_PATH / "documents.json") as f:
    documents = json.load(f)

with open(RAW_DATA_PATH / "queries.json") as f:
    queries = json.load(f)

print(documents[0])
print(queries[0])

{'id': 1, 'text': 'Anthropic develops AI systems with a focus on safety and alignment research.'}
{'query': 'What causes hallucinations in language models?', 'ground_truth_doc_id': 3}


### Step 2 — Document Embeddings

In this step, we generate **vector representations** (embeddings) for each document in our controlled corpus. These embeddings serve as the foundation for **semantic similarity search** and retrieval in the RAG pipeline.

Key objectives of this step:

- Convert raw text documents into numeric vectors using a configurable embedding model.
- Persist embeddings to disk for reuse, ensuring reproducibility.
- Validate the generated embeddings to ensure they are non-empty and correctly structured.

The embeddings produced here will be used in subsequent steps to construct a **FAISS index** for efficient retrieval.

#### Step 2.1 — Generate Document Embeddings

In this step, we generate vector embeddings for each document in our synthetic corpus using
the `EmbeddingModel` from `src/rag_eval/embeddings.py`. These embeddings will allow us
to perform similarity search and Retrieval-Augmented Generation (RAG) queries later.

The embeddings will first be attached to the `documents` in memory for inspection.

In [20]:
# Import our embedding abstraction
from rag_eval.embeddings import EmbeddingModel

# Initialize embedding model
embed_model = EmbeddingModel(model_name="text-embedding-3-small")

# Extract document texts
texts = [doc["text"] for doc in documents]

# Generate embeddings
embeddings_list = embed_model.embed_texts(texts)

# Attach embeddings to the documents (for inline inspection)
for doc, emb in zip(documents, embeddings_list):
    doc["embedding"] = emb

# Verify
documents[0]  # shows first doc with its embedding

{'id': 1,
 'text': 'Anthropic develops AI systems with a focus on safety and alignment research.',
 'embedding': [-0.0063036540523171425,
  0.022717000916600227,
  0.05662934482097626,
  0.023834025487303734,
  -0.003787212772294879,
  -0.01785983145236969,
  -0.010668205097317696,
  0.048395998775959015,
  0.03381193429231644,
  -0.01812339760363102,
  -0.010241476818919182,
  -0.06190069019794464,
  -0.012764194048941135,
  -0.04031326621770859,
  -0.011207890696823597,
  0.0034044126514345407,
  -0.011038454249501228,
  -0.017445653676986694,
  0.02650735154747963,
  -0.0371253527700901,
  -0.017746873199939728,
  0.008867161348462105,
  -0.0006302869878709316,
  0.047944169491529465,
  0.01683066226541996,
  -0.06822630763053894,
  0.008277271874248981,
  0.014948038384318352,
  0.020846927538514137,
  0.001415106002241373,
  0.055223651230335236,
  -0.027762435376644135,
  -0.015901900827884674,
  0.042321398854255676,
  -0.03235603868961334,
  0.037050049751996994,
  -0.021185800

#### Step 2.2 — Persist Embeddings to Disk

To avoid recomputing embeddings each time the notebook is run, we save the embeddings
to disk in Parquet format under `data/processed/`. This allows us to quickly load
embeddings in future runs or other experiments.

In [21]:
from pathlib import Path
import pandas as pd

# Ensure processed data directory exists
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
embeddings_file = PROCESSED_DATA_PATH / "embeddings.parquet"

# Create DataFrame and save
df_embeddings = pd.DataFrame({
    "id": [doc["id"] for doc in documents],
    "text": texts,
    "embedding": embeddings_list
})

df_embeddings.to_parquet(embeddings_file, index=False)

print(f"Saved embeddings for {len(documents)} documents to {embeddings_file}")

Saved embeddings for 4 documents to ../data/processed/embeddings.parquet


#### Step 2.3 — Validate Generated Embeddings

We validate that the embeddings were correctly generated and persisted without
assuming anything about the model used. The checks include:

1. The parquet file can be loaded successfully.
2. All expected columns exist: `id`, `text`, `embedding`.
3. Each embedding is a non-empty list.

This ensures that the persisted embeddings are usable for downstream retrieval and evaluation.

In [22]:
import pandas as pd
import numpy as np

# Load the persisted embeddings
embeddings_df = pd.read_parquet(embeddings_file)

# Check embeddings are non-empty sequences
for i, emb in enumerate(embeddings_df["embedding"]):
    if not isinstance(emb, (list, np.ndarray)):
        raise ValueError(f"Embedding at index {i} is not a list or ndarray")
    if len(emb) == 0:
        raise ValueError(f"Embedding at index {i} is empty")

print("✅ All embeddings are present and valid (non-empty sequences).")

✅ All embeddings are present and valid (non-empty sequences).


### Step 3 — Implement RAG Retrieval and Querying

In this phase, we will build a retrieval mechanism to support
Retrieval-Augmented Generation (RAG). This involves:

1. Creating a FAISS vector index from the document embeddings.
2. Running similarity search queries against the index.
3. Retrieving the most relevant documents for given user queries.

This step assumes that document embeddings have already been generated
and persisted in `data/processed/documents.parquet`.

#### Step 3.1 — Load Document Embeddings and Build FAISS Index

In this step, we will:

1. Load the persisted document embeddings from `data_processed_path`.
2. Initialize a FAISS vector index.
3. Populate the index with the embeddings so that we can perform
   similarity searches in later steps.

In [23]:
import faiss
import numpy as np

# Use existing embeddings_df from earlier step
embedding_matrix = np.vstack(embeddings_df["embedding"].values).astype("float32")

# Initialize FAISS index (L2 distance)
embedding_dim = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(embedding_dim)

# Add embeddings to index
index.add(embedding_matrix)

print(f"FAISS index created with {index.ntotal} vectors (dimension={embedding_dim}).")

FAISS index created with 4 vectors (dimension=1536).


#### Step 3.2 — Query Embedding and Retrieval

Now that we have a FAISS index of all document embeddings, we can embed a new query and perform similarity search
to retrieve the most relevant document(s). Here we demonstrate embedding a single query using `EmbeddingModel.embed_text`
and then retrieving the top matching document from the FAISS index.

In [24]:
# Example query
query = "What causes hallucinations in language models?"

# Embed the query using the new embed_text method
query_embedding = embed_model.embed_text(query)

# Retrieve top-1 similar document
D, I = index.search(np.array([query_embedding], dtype=np.float32), k=1)

# Map index to document
retrieved_doc = embeddings_df.iloc[I[0][0]]

print(f"Query: {query}")
print(f"Retrieved document ID: {retrieved_doc['id']}")
print(f"Retrieved document text: {retrieved_doc['text']}")


Query: What causes hallucinations in language models?
Retrieved document ID: 3
Retrieved document text: Hallucinations in large language models occur when the model generates unsupported or fabricated information.


#### Step 3.3 — Retrieve Top-k Relevant Documents for Each Query

In this step, we demonstrate how to retrieve the most relevant documents for a given query
based on their vector embeddings stored in the FAISS index. 

Key points:

- We retrieve **top-k matching documents** for each query rather than just a single document.
- Each query is first embedded using our `EmbeddingModel`.
- The FAISS index is then searched using the query embedding to return the most similar documents.
- This simulates a realistic Retrieval-Augmented Generation (RAG) workflow, where a single 
  query may have multiple relevant documents used to improve response quality.

We will loop over all queries in our dataset, embedding each query and retrieving the top-k
documents from the FAISS index for evaluation purposes.

In [25]:
# Number of top documents to retrieve per query
top_k = 2

# Extract query texts
query_texts = [q["query"] for q in queries]

# Generate embeddings for all queries at once
query_embeddings = embed_model.embed_texts(query_texts)

# Perform FAISS retrieval for each query
retrieval_results = []
for q_text, q_emb, q in zip(query_texts, query_embeddings, queries):
    # FAISS expects a 2D numpy array
    query_vector = np.array([q_emb], dtype=np.float32)
    D, I = index.search(query_vector, k=top_k)  # D = distances, I = indices
    
    # Map FAISS indices back to document IDs
    retrieved_docs = [documents[i]["id"] for i in I[0]]
    
    retrieval_results.append({
        "query": q_text,
        "retrieved_doc_ids": retrieved_docs,
        "ground_truth_doc_id": q["ground_truth_doc_id"]
    })

# Inspect the first retrieval result
retrieval_results[0]

{'query': 'What causes hallucinations in language models?',
 'retrieved_doc_ids': [3, 2],
 'ground_truth_doc_id': 3}

#### Step 3.4 — Validate Retrieval Results

In this step, we will check that our RAG retrieval logic is correctly finding the ground-truth documents for each query.  
Since our current synthetic corpus associates **one ground-truth document per query**, we will validate that the retrieved documents include the expected document IDs.  

This ensures that:

1. The FAISS vector search is working correctly.
2. Our embeddings are properly representing the semantic content of the documents.
3. The retrieval logic is ready to be extended to multiple queries or larger corpora.

In [26]:
# Perform retrieval for all queries and validate
query_texts = [q["query"] for q in queries]
query_embeddings = embed_model.embed_texts(query_texts)

retrieval_results = []

for query_text, query_emb, q in zip(query_texts, query_embeddings, queries):
    # FAISS expects a 2D numpy array
    query_vector = np.array([query_emb], dtype="float32")
    
    # Retrieve top k documents (here k=1 to match our synthetic setup)
    D, I = index.search(query_vector, k=1)
    
    retrieved_doc_id = int(embeddings_df.iloc[I[0][0]]["id"])
    
    retrieval_results.append({
        "query": query_text,
        "retrieved_doc_id": retrieved_doc_id,
        "ground_truth_doc_id": q["ground_truth_doc_id"]
    })

# Convert to DataFrame for easier inspection
retrieval_df = pd.DataFrame(retrieval_results)

# Assert all queries retrieved the expected document
for i, row in retrieval_df.iterrows():
    assert row["retrieved_doc_id"] == row["ground_truth_doc_id"], (
        f"Query '{row['query']}' expected doc {row['ground_truth_doc_id']}, "
        f"got {row['retrieved_doc_id']}"
    )

print("✅ All queries correctly retrieved their ground-truth documents.")
retrieval_df

✅ All queries correctly retrieved their ground-truth documents.


,query,retrieved_doc_id,ground_truth_doc_id
0,What causes hallucinations in language models?,3,3
1,How does RAG improve factual accuracy?,2,2
2,What is the role of vector databases in RAG?,4,4


### Step 4 — Evaluate Retrieval Performance

So far, we have validated that our RAG retrieval pipeline functions correctly:
documents are embedded, indexed in FAISS, and successfully retrieved
for a given query.

In this step, we move from validation to evaluation.

Rather than relying solely on assertions, we will compute quantitative
metrics to measure retrieval performance. This shifts the notebook from
a functional prototype toward a minimal evaluation framework.

We will:

- Compute Retrieval Accuracy (exact match).
- Support configurable top-k retrieval.
- Summarize evaluation results.

This establishes the foundation for more advanced retrieval metrics
in future iterations of the framework.

#### Step 4.1 — Compute Retrieval Accuracy

In this sub-step, we calculate the retrieval accuracy of our RAG pipeline:

- For each query, we check whether the top retrieved document matches the
  ground-truth document.
- We report the **Retrieval Accuracy @ 1**, i.e., the fraction of queries
  for which the first retrieved document is correct.
- This metric provides a simple but clear indication of the system's
  ability to find the correct document for a given query.

Later, we can extend this to top-k retrieval metrics or other evaluation
criteria, but for now we focus on this minimal setup.


In [27]:
# Configurable number of retrieved documents
TOP_K = 1

# If TOP_K > 1, ensure retrieval logic uses this value in Step 3.3 / 3.4
# (For now, we assume TOP_K = 1 for exact-match evaluation.)

# Compute Retrieval Accuracy (Exact Match)
accuracy = (retrieval_df["retrieved_doc_id"] == retrieval_df["ground_truth_doc_id"]).mean()


#### Step 4.2 — Evaluation Summary

We have successfully:

- Generated document embeddings and stored them persistently.
- Built a FAISS vector index for similarity search.
- Embedded evaluation queries.
- Retrieved top-k relevant documents for each query.
- Computed retrieval accuracy against ground-truth labels.

This notebook now serves as a minimal, reproducible RAG retrieval
evaluation prototype. It validates that:

- The embedding model is functioning correctly.
- The FAISS index is constructed properly.
- Retrieval logic maps correctly back to document IDs.
- Evaluation metrics can be computed programmatically.

Below, we display the computed retrieval accuracy as a single summary
statement to highlight overall performance.


In [28]:
print(f"Retrieval Accuracy @ {TOP_K}: {accuracy:.2%}")

Retrieval Accuracy @ 1: 100.00%


## Conclusion

This notebook demonstrates a **minimal RAG retrieval evaluation pipeline**:

- Document embeddings are generated using a configurable embedding model.
- Embeddings are persisted and used to construct a FAISS vector index.
- Evaluation queries are embedded, and the most relevant documents are retrieved.
- Retrieval accuracy is computed against ground-truth labels.

This prototype validates the core functionality of:

- Embedding generation
- Vector similarity search
- Retrieval mapping
- Evaluation metric computation

The notebook is now ready to serve as a **reproducible baseline** for further
development, such as extracting retrieval and evaluation logic into
reusable framework components.
